# Abgabe 2 – Kapitel 4, Aufgabe 12: Softmax-Regression von Grund auf

Batch-Gradientenverfahren mit Early Stopping für die Softmax-Regression, nur mit NumPy, auf dem Iris-Datensatz.

Die theoretischen Aufgaben stehen in `Kapitel_4.md`.

## 1. Daten laden

Zwei Merkmale (Kronblattlänge und -breite), drei Klassen.

In [16]:
import numpy as np
from sklearn import datasets

iris = datasets.load_iris()
X = iris.data[:, 2:]   # Kronblattlänge und -breite
y = iris.target        # 0 = Setosa, 1 = Versicolor, 2 = Virginica
print('X:', X.shape, '| Klassen:', np.unique(y))

X: (150, 2) | Klassen: [0 1 2]


## 2. Bias-Term hinzufügen

Konstante 1 je Datenpunkt für den Achsenabschnitt.

In [17]:
X_with_bias = np.c_[np.ones(len(X)), X]

## 3. Aufteilung in Training, Validierung und Test

Manueller Split: 60 % Training, 20 % Validierung, 20 % Test.

In [18]:
np.random.seed(2042)

test_ratio = 0.2
validation_ratio = 0.2
total_size = len(X_with_bias)

test_size = int(total_size * test_ratio)
validation_size = int(total_size * validation_ratio)
train_size = total_size - test_size - validation_size

rnd_indices = np.random.permutation(total_size)

X_train = X_with_bias[rnd_indices[:train_size]]
y_train = y[rnd_indices[:train_size]]
X_valid = X_with_bias[rnd_indices[train_size:train_size + validation_size]]
y_valid = y[rnd_indices[train_size:train_size + validation_size]]
X_test = X_with_bias[rnd_indices[-test_size:]]
y_test = y[rnd_indices[-test_size:]]

print('Train:', len(X_train), '| Valid:', len(X_valid), '| Test:', len(X_test))

Train: 90 | Valid: 30 | Test: 30


## 4. One-Hot-Kodierung der Zielwerte

Klassen als One-Hot-Vektoren.

In [19]:
def to_one_hot(y):
    n_classes = y.max() + 1
    m = len(y)
    Y_one_hot = np.zeros((m, n_classes))
    Y_one_hot[np.arange(m), y] = 1
    return Y_one_hot

Y_train_one_hot = to_one_hot(y_train)
Y_valid_one_hot = to_one_hot(y_valid)
Y_test_one_hot = to_one_hot(y_test)
Y_train_one_hot[:3]

array([[1., 0., 0.],
       [0., 1., 0.],
       [0., 0., 1.]])

## 5. Merkmale standardisieren

Mittelwert und Standardabweichung aus dem Trainingsset, auf alle Mengen angewendet (ohne Bias-Term).

In [20]:
mean = X_train[:, 1:].mean(axis=0)
std = X_train[:, 1:].std(axis=0)
X_train[:, 1:] = (X_train[:, 1:] - mean) / std
X_valid[:, 1:] = (X_valid[:, 1:] - mean) / std
X_test[:, 1:] = (X_test[:, 1:] - mean) / std

## 6. Softmax-Funktion

Wandelt die Logits in Wahrscheinlichkeiten um:



In [21]:
def softmax(logits):
    exps = np.exp(logits)
    exp_sums = exps.sum(axis=1, keepdims=True)
    return exps / exp_sums

## 7. Training: Batch-Gradientenverfahren mit Early Stopping

Kreuzentropie mit L2-Regularisierung



Abbruch, sobald der Validierungsverlust nicht mehr sinkt.

In [22]:
n_inputs = X_train.shape[1]              
n_outputs = len(np.unique(y_train))      

eta = 0.5            
n_epochs = 5001
m = len(X_train)
epsilon = 1e-5       
alpha = 0.01        

np.random.seed(42)
Theta = np.random.randn(n_inputs, n_outputs)

best_loss = np.inf
best_Theta = Theta

for epoch in range(n_epochs):
  
    logits = X_train @ Theta
    Y_proba = softmax(logits)

  
    logits_valid = X_valid @ Theta
    Y_proba_valid = softmax(logits_valid)
    xentropy_loss = -np.mean(np.sum(Y_valid_one_hot * np.log(Y_proba_valid + epsilon), axis=1))
    l2_loss = 0.5 * np.sum(np.square(Theta[1:]))   # Bias nicht regularisieren
    loss = xentropy_loss + alpha * l2_loss

  
    error = Y_proba - Y_train_one_hot
    gradients = (1 / m) * X_train.T @ error
    gradients += np.r_[np.zeros((1, n_outputs)), alpha * Theta[1:]]
    Theta = Theta - eta * gradients

    if epoch % 500 == 0:
        print('Epoche {:>4}  Validierungsverlust: {:.5f}'.format(epoch, loss))

   
    if loss < best_loss:
        best_loss = loss
        best_Theta = Theta
    else:
        print('Epoche {:>4}  Verlust steigt ({:.5f}) – Early Stopping'.format(epoch, loss))
        break

print('\nBestes Theta:\n', best_Theta)

Epoche    0  Validierungsverlust: 3.49609
Epoche  297  Verlust steigt (0.30462) – Early Stopping

Bestes Theta:
 [[-0.05729786  1.97470314 -0.91126689]
 [-2.14969994  0.22987802  2.15783466]
 [-1.88275694 -0.24006941  2.54642963]]


## 8. Auswertung

Genauigkeit auf Validierungs- und Testset mit dem besten $\Theta$.

In [23]:
def predict(X, Theta):
    return softmax(X @ Theta).argmax(axis=1)

acc_valid = np.mean(predict(X_valid, best_Theta) == y_valid)
acc_test = np.mean(predict(X_test, best_Theta) == y_test)
print('Validierungsgenauigkeit: {:.2%}'.format(acc_valid))
print('Testgenauigkeit:         {:.2%}'.format(acc_test))

Validierungsgenauigkeit: 96.67%
Testgenauigkeit:         96.67%


Das Modell erreicht auf Iris rund 90–100 % Genauigkeit.